In [ ]:
!pip -q install -U "huggingface_hub[cli]"

In [10]:
from google.colab import drive
import os
drive.mount('/content/drive')

ROOT_DIR = "/content/drive/MyDrive/AI_Related_Study/CadCoder"
MODEL_DIR = f"{ROOT_DIR}/model"
OUTPUT_DIR = f"{ROOT_DIR}/outout"
DATA_DIR = f"{ROOT_DIR}/llava_data"
HF_CACHE = f"{ROOT_DIR}/hf_cache"

os.makedirs(MODEL_DIR, exist_ok=True)
print("MODEL_DIR =", MODEL_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)
os.makedirs(DATA_DIR, exist_ok=True)
print(DATA_DIR)
os.makedirs(HF_CACHE, exist_ok=True)
print(HF_CACHE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL_DIR = /content/drive/MyDrive/AI_Related_Study/CadCoder/model
OUTPUT_DIR = /content/drive/MyDrive/AI_Related_Study/CadCoder/outout
/content/drive/MyDrive/AI_Related_Study/CadCoder/llava_data
/content/drive/MyDrive/AI_Related_Study/CadCoder/hf_cache


In [ ]:
!git clone https://github.com/anniedoris/CAD-Coder.git {ROOT_DIR}/CAD-Coder

### 事前学習モデルの事前ダウンロード（毎回ダウンロードはしない）

In [ ]:
!pip -q install -U "huggingface_hub[cli]"

# 遅い/不安定なときの保険
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

# まずローカルSSD側へ
!rm -rf /content/CAD-Coder-model
!hf download CADCODER/CAD-Coder --local-dir /content/CAD-Coder-model

Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Still waiting to acquire lock on /content/CAD-Coder-model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 13 files: 100% 13/13 [05:18<00:00, 24.47s/it]
Download complete: : 26.7GB [05:18, 176MB/s]                              /content/CAD-Coder-model
Download complete: : 26.7GB [05:18, 83.9MB/s]


### 事前学習モデルの事前ダウンロード（毎回ダウンロードはしない）

In [ ]:
!ls -lh /content/CAD-Coder-model/model-*.safetensors
!ls -lh /content/CAD-Coder-model/model.safetensors.index.json

-rw-r--r-- 1 root root 4.7G Mar  4 19:37 /content/CAD-Coder-model/model-00001-of-00006.safetensors
-rw-r--r-- 1 root root 4.7G Mar  4 19:37 /content/CAD-Coder-model/model-00002-of-00006.safetensors
-rw-r--r-- 1 root root 4.7G Mar  4 19:38 /content/CAD-Coder-model/model-00003-of-00006.safetensors
-rw-r--r-- 1 root root 4.6G Mar  4 19:37 /content/CAD-Coder-model/model-00004-of-00006.safetensors
-rw-r--r-- 1 root root 4.6G Mar  4 19:37 /content/CAD-Coder-model/model-00005-of-00006.safetensors
-rw-r--r-- 1 root root 1.8G Mar  4 19:35 /content/CAD-Coder-model/model-00006-of-00006.safetensors
-rw-r--r-- 1 root root 78K Mar  4 19:32 /content/CAD-Coder-model/model.safetensors.index.json


In [ ]:
!mkdir -p {ROOT_DIR}/model
!rm -rf {ROOT_DIR}/model/CAD-Coder
!cp -a {ROOT_DIR}/model/CAD-Coder

### 寸法指定のPROMPTを試す

In [ ]:
%%bash
set -e
cd "/content/drive/MyDrive/AI_Related_Study/CadCoder/CAD-Coder"
#mkdir -p inference

cat > inference/my_test.jsonl <<'JSONL'
{"question_id": 1, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 2, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words. Target overall dimensions (millimeters): length 670, width 250, thickness 33. Please match the bounding box as closely as possible.", "category": "default", "ground_truth": ""}
{"question_id": 3, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words. Target overall dimensions (millimeters): length 70, width 30, thickness 3. Please match the bounding box as closely as possible.", "category": "default", "ground_truth": ""}
{"question_id": 4, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words. Target overall dimensions (millimeters): length 950, width 370, thickness 40. Hint: use standard CadQuery primitives (e.g., cq.Workplane('XY').box(L, W, T) OR cq.Workplane('XY').rect(L, W).extrude(T)) and assign the final result to variable `solid`.", "category": "default", "ground_truth": ""}
JSONL

wc -l inference/my_test.jsonl
tail -n 4 inference/my_test.jsonl

4 inference/my_test.jsonl
{"question_id": 1, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 2, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words. Target overall dimensions (millimeters): length 670, width 250, thickness 33. Please match the bounding box as closely as possible.", "category": "default", "ground_truth": ""}
{"question_id": 3, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words. Target overall dimensions (millimeters): length 70, width 30, thickness 3. Please match the bounding box as closely as possible.", "category": "default", "ground_truth": ""}
{"question_id": 4, "image": "00675497_0.png", "text": "Generate the CADQuery code needed to 

## 自分の画像で試してみる(CADのような画像01~12.pngと実写のa-e.pngで推論実施します。あらかじめ画像をinference/test100_imagesに保存しておいてください)

In [11]:
%%bash
set -e
cd "/content/drive/MyDrive/AI_Related_Study/CadCoder/CAD-Coder"
#mkdir -p inference
cat > inference/my_images.jsonl <<'JSONL'
{"question_id": 1, "image": "01.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 2, "image": "02.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 3, "image": "03.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 4, "image": "04.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 5, "image": "05.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 6, "image": "06.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 7, "image": "07.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 8, "image": "08.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 9, "image": "09.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 10, "image": "10.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 11, "image": "11.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 12, "image": "12.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 13, "image": "a.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 14, "image": "b.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 15, "image": "c.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 16, "image": "d.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 17, "image": "e.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
JSONL
wc -l inference/my_images.jsonl
tail -n 4 inference/my_images.jsonl

17 inference/my_images.jsonl
{"question_id": 14, "image": "b.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 15, "image": "c.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 16, "image": "d.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}
{"question_id": 17, "image": "e.png", "text": "Generate the CADQuery code needed to create the CAD for the provided image. Just the code, no other words.", "category": "default", "ground_truth": ""}


### 学習データの事前ダウンロード

In [ ]:
%env HF_HOME={HF_CACHE}

!mkdir -p {DATA_DIR}/llava_pretrain_data

!hf download liuhaotian/LLaVA-Pretrain \
  --repo-type dataset \
  --local-dir {DATA_DIR}/llava_pretrain_data

env: HF_HOME=/content/drive/MyDrive/AI_Related_Study/CadCoder/hf_cache
Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Fetching 5 files: 100% 5/5 [00:02<00:00,  2.13it/s]
Download complete: : 0.00B [00:02, ?B/s]              /content/drive/MyDrive/AI_Related_Study/CadCoder/llava_data/llava_pretrain_data
Download complete: : 0.00B [00:02, ?B/s]


In [ ]:
# unizp めちゃくちゃ時間かかります。
!unzip -q {DATA_DIR}/llava_pretrain_data/images.zip \
  -d {DATA_DIR}/llava_pretrain_data

replace /content/drive/MyDrive/AI_Related_Study/CadCoder/llava_data/llava_pretrain_data/00001/000015879.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
N
None


## 学習環境のセットアップ

In [ ]:
%cd {ROOT_DIR}/CAD-Coder

/content/drive/MyDrive/AI_Related_Study/CadCoder/CAD-Coder


In [ ]:
# LLaVA から取得（zero2 / zero3）
!curl -L https://raw.githubusercontent.com/haotian-liu/LLaVA/main/scripts/zero2.json -o scripts/zero2.json
!curl -L https://raw.githubusercontent.com/haotian-liu/LLaVA/main/scripts/zero3.json -o scripts/zero3.json
# JSONとして壊れてないかチェック
!python -m json.tool scripts/zero2.json >/dev/null && echo "zero2.json OK"
!python -m json.tool scripts/zero3.json >/dev/null && echo "zero3.json OK"


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   556  100   556    0     0   1011      0 --:--:-- --:--:-- --:--:--  1010
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   801  100   801    0     0   1628      0 --:--:-- --:--:-- --:--:--  1631
zero2.json OK
zero3.json OK


In [ ]:
!ls -l {DATA_DIR}/llava_pretrain_data/blip_laion_cc_sbu_558k.json
!ls -d {DATA_DIR}/llava_pretrain_data/00000 2>/dev/null || ls -d $LLAVA_DATA_ROOT/llava_pretrain_data/images 2>/dev/null


-rw------- 1 root root 180506011 Mar  4 21:22 /content/drive/MyDrive/AI_Related_Study/CadCoder/llava_data/llava_pretrain_data/blip_laion_cc_sbu_558k.json


In [ ]:
%cp {ROOT_DIR}/CAD-Coder/llava/train/train_mem.py {ROOT_DIR}/CAD-Coder/llava/train/train_mem.py_org
!sed -i 's/flash_attention_2/sdpa/g' {ROOT_DIR}/CAD-Coder/llava/train/train_mem.py
!grep -n "attn_implementation" -n {ROOT_DIR}/CAD-Coder/llava/train/train_mem.py

4:    train(attn_implementation="sdpa")


In [ ]:
%cd {ROOT_DIR}/CAD-Coder

/content/drive/MyDrive/AI_Related_Study/CadCoder/CAD-Coder


In [ ]:
%rm scripts/v1_5/phase1_cadcoder_1gpu.sh

In [ ]:
!cp scripts/v1_5/phase1_cadcoder.sh scripts/v1_5/phase1_cadcoder_1gpu.sh
# 64 → 4 に下げる（まず動かす）
!sed -i 's/--per_device_train_batch_size 64/--per_device_train_batch_size 4/g' scripts/v1_5/phase1_cadcoder_1gpu.sh
# accum 1 → 16（実効64に近づける）
!sed -i 's/--gradient_accumulation_steps 1/--gradient_accumulation_steps 16/g' scripts/v1_5/phase1_cadcoder_1gpu.sh
# workerも少し減らす（安定用）
!sed -i 's/--dataloader_num_workers 4/--dataloader_num_workers 2/g' scripts/v1_5/phase1_cadcoder_1gpu.sh
!grep -n "per_device_train_batch_size\|gradient_accumulation_steps\|dataloader_num_workers" scripts/v1_5/phase1_cadcoder_1gpu.sh

20:    --per_device_train_batch_size 4 \
22:    --gradient_accumulation_steps 16 \
35:    --dataloader_num_workers 2 \


In [ ]:
%mkdir -p {OUTPUT_DIR}/checkpoints/cadcoder_stage1
%mkdir -p {OUTPUT_DIR}/checkpoints/cadcoder_stage1/logs